# Gaussians & noise: hands-on (optional)
**LLMs From Basics · Lesson P5.1**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (or a ✓) means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
print("ready")

## 1 · Sample any Gaussian from a standard normal
The reparameterization trick: if $\varepsilon \sim \mathcal N(0, 1)$, then $x = \mu + \sigma\varepsilon \sim \mathcal N(\mu, \sigma^2)$.

Implement `sample_normal(mu, sigma, n, rng)` using **only** `rng.standard_normal` (no `rng.normal`). `sigma` is the standard deviation, not the variance.

In [ ]:
def sample_normal(mu, sigma, n, rng):
    # TODO: draw n standard normals and shift/scale them
    raise NotImplementedError("Exercise 1")

**Check 1**

In [ ]:
x = sample_normal(170.0, 10.0, 200_000, np.random.default_rng(1))
assert x.shape == (200_000,)
assert abs(x.mean() - 170) < 0.1, x.mean()
assert abs(x.std() - 10) < 0.1, x.std()
# same seed, same draws: the randomness lives entirely in epsilon
a = sample_normal(0.0, 1.0, 5, np.random.default_rng(7))
b = sample_normal(3.0, 2.0, 5, np.random.default_rng(7))
assert np.allclose(b, 3 + 2 * a)
print(f"✓ exercise 1:  mean {x.mean():.2f}, std {x.std():.2f}")

## 2 · The 68–95–99.7 rule, by sampling
Implement `frac_within(x, mu, sigma, k)`: the fraction of the samples `x` that lie within $k$ standard deviations of $\mu$, i.e. $|x - \mu| < k\sigma$.

In [ ]:
def frac_within(x, mu, sigma, k):
    # TODO
    raise NotImplementedError("Exercise 2")

**Check 2**

In [ ]:
z = np.random.default_rng(2).standard_normal(1_000_000)
f1, f2, f3 = (frac_within(z, 0, 1, k) for k in (1, 2, 3))
assert abs(f1 - 0.6827) < 0.003, f1
assert abs(f2 - 0.9545) < 0.002, f2
assert abs(f3 - 0.9973) < 0.001, f3
assert frac_within(np.array([0.0, 0.5, 2.0, -3.0]), 0, 1, 1) == 0.5
print(f"✓ exercise 2:  {f1:.4f}, {f2:.4f}, {f3:.4f}  (theory 0.6827, 0.9545, 0.9973)")

## 3 · Variances add
For independent $X \sim \mathcal N(\mu_1, \sigma_1^2)$ and $Y \sim \mathcal N(\mu_2, \sigma_2^2)$, $X + Y \sim \mathcal N(\mu_1 + \mu_2, \sigma_1^2 + \sigma_2^2)$.

1. Implement `std_of_sum(s1, s2)`: the standard deviation of $X + Y$ (from the formula, no sampling).
2. Then confirm it empirically: set `empirical` to the standard deviation of 500,000 sampled sums with $\sigma_1 = 3$, $\sigma_2 = 4$ (use your `sample_normal`).

In [ ]:
def std_of_sum(s1, s2):
    # TODO
    raise NotImplementedError("Exercise 3")

empirical = None   # TODO: std of 500,000 sampled X + Y with sigma 3 and 4

**Check 3**

In [ ]:
assert np.isclose(std_of_sum(3, 4), 5.0)
assert np.isclose(std_of_sum(0.3, 0.4), 0.5)
assert empirical is not None and abs(empirical - 5.0) < 0.03, empirical
print(f"✓ exercise 3:  formula 5.000, sampled {empirical:.3f}  (not 3 + 4 = 7)")

## 4 · Forward-noise a 2D dataset
Here is a spiral of 2,000 points, scaled so each coordinate has mean 0 and variance 1.

Implement `noisy(x0, abar, eps)` $= \sqrt{\bar\alpha}\,x_0 + \sqrt{1 - \bar\alpha}\,\varepsilon$ and `snr(abar)` $= \bar\alpha / (1 - \bar\alpha)$. Then the check verifies the process is **variance-preserving**.

In [ ]:
def make_spiral(n, rng):
    th = np.linspace(0.6 * np.pi, 4 * np.pi, n)
    pts = np.stack([th * np.cos(th), th * np.sin(th)], axis=1) + 0.25 * rng.standard_normal((n, 2))
    return (pts - pts.mean(0)) / pts.std(0)

x0 = make_spiral(2000, np.random.default_rng(4))
print(x0.shape, x0.mean(0).round(3), x0.var(0).round(3))

In [ ]:
def noisy(x0, abar, eps):
    # TODO
    raise NotImplementedError("Exercise 4a")

def snr(abar):
    # TODO
    raise NotImplementedError("Exercise 4b")

**Check 4**

In [ ]:
eps = np.random.default_rng(5).standard_normal(x0.shape)
assert np.allclose(noisy(x0, 1.0, eps), x0)            # t = 0: clean data
assert np.allclose(noisy(x0, 0.0, eps), eps)           # t = T: pure noise
assert np.isclose(noisy(np.array(2.0), 0.64, np.array(-1.0)), 1.0)   # the lesson's worked example
for abar in [0.9, 0.5, 0.1, 0.01]:
    v = noisy(x0, abar, eps).var(0)
    assert np.all(np.abs(v - 1) < 0.08), (abar, v)
assert np.isclose(snr(0.5), 1.0) and np.isclose(snr(0.64), 0.64 / 0.36)
print("✓ exercise 4:  variance stays ≈ 1 at every noise level")

## 5 · Two noise schedules
DDPM (Ho et al., 2020) used $T = 1000$ steps with $\beta_t$ rising linearly from $10^{-4}$ to $0.02$ and $\bar\alpha_t = \prod_{s=1}^{t}(1 - \beta_s)$. The cosine schedule (Nichol & Dhariwal, 2021) sets $\bar\alpha_t = f(t)/f(0)$ with $f(t) = \cos^2\!\left(\frac{t/T + s}{1 + s}\cdot\frac{\pi}{2}\right)$, $s = 0.008$.

Implement both, returning arrays of length $T + 1$ for $t = 0, \dots, T$ (so `abar[0] == 1`).

In [ ]:
def abar_linear(T=1000, b0=1e-4, b1=0.02):
    # TODO: betas for t = 1..T, cumulative product, then prepend 1.0 for t = 0
    raise NotImplementedError("Exercise 5a")

def abar_cosine(T=1000, s=0.008):
    # TODO
    raise NotImplementedError("Exercise 5b")

**Check 5**

In [ ]:
lin, cos = abar_linear(), abar_cosine()
assert lin.shape == cos.shape == (1001,)
assert lin[0] == 1.0 and np.isclose(cos[0], 1.0)
assert np.all(np.diff(lin) < 0) and np.all(np.diff(cos) <= 0)
assert abs(lin[-1] - 4.04e-5) < 1e-6, lin[-1]
assert cos[-1] < 1e-6
assert abs(lin[500] - 0.0786) < 1e-3 and abs(cos[500] - 0.4938) < 1e-3
half_lin, half_cos = int(np.argmax(lin < 0.5)), int(np.argmax(cos < 0.5))
print(f"✓ exercise 5:  SNR drops below 1 at t = {half_lin} (linear) vs t = {half_cos} (cosine)")

## Experiment · watch the spiral dissolve
Run this to see the forward process at a few steps for both schedules. Then try: a different dataset (two blobs?), a fresh `eps`, or plotting `np.log(snr(abar))` against $t$ for both schedules. The cosine schedule's log-SNR falls almost linearly over the middle of the range, which is one way to see why it "destroys information more evenly".

In [ ]:
steps = [0, 100, 250, 400, 600, 1000]
fig, axes = plt.subplots(2, len(steps), figsize=(2.2 * len(steps), 4.6))
for row, (name, ab) in enumerate([("linear", lin), ("cosine", cos)]):
    for ax, t in zip(axes[row], steps):
        xt = noisy(x0, ab[t], eps)
        ax.scatter(xt[:, 0], xt[:, 1], s=1)
        ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{name} t={t}", fontsize=9)
plt.tight_layout(); plt.show()

## Recap
- $x = \mu + \sigma\varepsilon$ turns standard normals into any Gaussian, and keeps the randomness separate from the parameters.
- About 68%, 95% and 99.7% of draws fall within 1, 2 and 3 standard deviations; you measured it.
- Independent Gaussians: **variances add** ($3$ and $4$ give $5$).
- $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$ keeps unit variance while the SNR $\bar\alpha_t/(1-\bar\alpha_t)$ falls; the schedule decides how fast.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def sample_normal(mu, sigma, n, rng):
    eps = rng.standard_normal(n)
    return mu + sigma * eps

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def frac_within(x, mu, sigma, k):
    return float(np.mean(np.abs(x - mu) < k * sigma))

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def std_of_sum(s1, s2):
    return float(np.sqrt(s1 ** 2 + s2 ** 2))

r = np.random.default_rng(3)
empirical = float(np.std(sample_normal(0, 3, 500_000, r) + sample_normal(0, 4, 500_000, r)))

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def noisy(x0, abar, eps):
    return np.sqrt(abar) * x0 + np.sqrt(1 - abar) * eps

def snr(abar):
    return abar / (1 - abar)

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def abar_linear(T=1000, b0=1e-4, b1=0.02):
    betas = np.linspace(b0, b1, T)
    return np.concatenate([[1.0], np.cumprod(1 - betas)])

def abar_cosine(T=1000, s=0.008):
    t = np.arange(T + 1)
    f = np.cos((t / T + s) / (1 + s) * np.pi / 2) ** 2
    return f / f[0]